In [59]:
import numpy as np

In [60]:
data=[
    "I love this movie",
    "This movie is amazing",
    "I hate this movie",
    "The story was boring",
    "I loved It"
]

<h2>Tokenization</h2>

In [61]:
tokens=[]

In [62]:
for d in data:
    tokens.append(d.lower().split(" "))

In [63]:
tokens

[['i', 'love', 'this', 'movie'],
 ['this', 'movie', 'is', 'amazing'],
 ['i', 'hate', 'this', 'movie'],
 ['the', 'story', 'was', 'boring'],
 ['i', 'loved', 'it']]

<h2>Created Vocabulary Set</h2>

In [64]:
vocabularySet=set()

In [65]:
for data in tokens:
    for d in data:
        vocabularySet.add(d)

In [66]:
vocabularySet

{'amazing',
 'boring',
 'hate',
 'i',
 'is',
 'it',
 'love',
 'loved',
 'movie',
 'story',
 'the',
 'this',
 'was'}

In [67]:
len(vocabularySet)

13

<h2>Vocabulary Indexing</h2>

In [68]:
words_to_id={}

In [69]:
for index,data in enumerate(vocabularySet):
    words_to_id[data]=index+2

In [70]:
words_to_id

{'movie': 2,
 'it': 3,
 'love': 4,
 'this': 5,
 'amazing': 6,
 'hate': 7,
 'i': 8,
 'is': 9,
 'the': 10,
 'story': 11,
 'was': 12,
 'loved': 13,
 'boring': 14}

<h2>Sentence to numbers</h2>

In [71]:
sentence_to_num=[]
for sentence in tokens:
    temp=[]
    for words in sentence:
       temp.append(words_to_id[words])

    sentence_to_num.append(temp)

In [72]:
sentence_to_num

[[8, 4, 5, 2], [5, 2, 9, 6], [8, 7, 5, 2], [10, 11, 12, 14], [8, 13, 3]]

<h2>Padding</h2>

In [73]:
maxLen=len(max(sentence_to_num))

In [74]:
padded=[]
for x in sentence_to_num:
    number_of_zeros=maxLen-len(x)
    padded.append(x+[0]*number_of_zeros)

In [75]:
padded

[[8, 4, 5, 2], [5, 2, 9, 6], [8, 7, 5, 2], [10, 11, 12, 14], [8, 13, 3, 0]]

<h2>Embeddings</h2>

In [76]:
import torch

In [77]:
embedding=torch.nn.Embedding(len(vocabularySet)+2,4,padding_idx=0)

In [78]:
padded_tensor=torch.tensor(padded,dtype=torch.long)

In [79]:
padded_tensor

tensor([[ 8,  4,  5,  2],
        [ 5,  2,  9,  6],
        [ 8,  7,  5,  2],
        [10, 11, 12, 14],
        [ 8, 13,  3,  0]])

In [80]:
embedding.weight.shape

torch.Size([15, 4])

In [81]:
embedded=embedding(padded_tensor)

In [82]:
embedded.shape

torch.Size([5, 4, 4])

<h2>Connecting to LSTM layer</h2>

In [83]:
lstm_layer=torch.nn.LSTM(input_size=4,hidden_size=8,batch_first=True)

In [84]:
output_lstm,hiden_lstm=lstm_layer(embedded)

In [85]:
print(f'Output_Shape:{output_lstm.shape}\nHidden:{hiden_lstm[0].shape}\nCell :{hiden_lstm[1].shape}')

Output_Shape:torch.Size([5, 4, 8])
Hidden:torch.Size([1, 5, 8])
Cell :torch.Size([1, 5, 8])


In [86]:
final_hidden_state=hiden_lstm[0].squeeze(0)

In [87]:
final_hidden_state.shape

torch.Size([5, 8])

In [88]:
linear=torch.nn.Linear(8,1)

In [89]:
target=torch.tensor([1,1,0,0,1],dtype=torch.float32)
target=target.reshape(5,-1)

In [90]:
loss_fn=torch.nn.BCEWithLogitsLoss()

In [91]:
optimizer=torch.optim.Adam(list(lstm_layer.parameters())+list(linear.parameters())+list(embedding.parameters()),lr=0.001)

In [92]:
for epoch in range(1001):
    optimizer.zero_grad()

    # Forward pass
    embedded = embedding(padded_tensor)

    output_lstm, (hidden_lstm, cell_lstm) = lstm_layer(embedded)

    final_hidden_state = hidden_lstm[0].squeeze(0)

    prediction = linear(final_hidden_state)

    # Loss
    loss = loss_fn(prediction, target)

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Loss: {loss.item():.4f}")

Loss: 0.6918
Loss: 0.5918
Loss: 0.3142
Loss: 0.0997
Loss: 0.0464
Loss: 0.0270
Loss: 0.0178
Loss: 0.0127
Loss: 0.0096
Loss: 0.0075
Loss: 0.0060


<h2>Input Authentication for testing</h2>

1)Tokenize
2)give appropriant int id if unknow then 1
3) if len of token array shorter then maxlen then pad if more then truncate
4) feed to model to predict

In [121]:
def inputAutheticator(input):
    tempToken=input.lower().split(" ")

    tempSequence=[]
    for word in tempToken:
        if word in words_to_id:
            tempSequence.append(words_to_id[word])
        else:
            tempSequence.append(1)

    paddedTempSequence=[]
    if len(tempSequence)>maxLen:
        paddedTempSequence=tempSequence[0:maxLen]
    elif len(tempSequence)<maxLen:
        paddedTempSequence=(tempSequence+[0]*(maxLen-len(tempSequence)))
    else:
        paddedTempSequence=tempSequence

    return paddedTempSequence


In [126]:
print(inputAutheticator("This movie was fantastic"))

[5, 2, 12, 1]
